# VitalDB cohort selection

Retrieves the public track inventory, screens modality availability, and assesses candidate surgeries. Requires authorized VitalDB access and local storage; see ../docs/DATA_ACCESS.md.

Run the setup cell first. Full processing and training require locally provisioned datasets; the quick start uses synthetic tests and saved aggregate results.


In [ ]:
# Load the libraries and project settings needed for this step.
import vitaldb
import pandas as pd
import numpy as np
from pathlib import Path

#Cohort test for n cases
N_CASES = 10


# 1. Project paths


# Create the destination directory before writing any outputs.
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
MANIFEST_DIR = PROJECT / "data" / "manifests"

# Create the destination directory before writing any outputs.
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


# 2. Signal settings


WAVEFORM_TRACKS = ["ART", "ECG_II", "PLETH"]
MAP_TRACK = ["ART_MBP"]

WAVEFORM_FS = 100          # Hz
MAP_INTERVAL = 1           # seconds


# 3. Temporary / exploratory inclusion thresholds

# These are NOT final study inclusion criteria yet.
# They are simply useful for screening cases while the
# data pipeline is being developed.


MIN_DURATION_MINUTES = 30

MAX_ART_MISSING_PCT = 20
MAX_ECG_MISSING_PCT = 20
MAX_PPG_MISSING_PCT = 20

# MAP is sparse by design, so do NOT require <20% missingness.
# Instead require reasonable measurement density and temporal coverage.

MIN_MAP_MEASUREMENTS = 300
MIN_MAP_MEASUREMENTS_PER_MINUTE = 10
MIN_MAP_COVERAGE_SPAN_PCT = 80
MAX_MAP_MEDIAN_GAP_SEC = 10


# 4. Find cases containing the waveform tracks


eligible_ids = vitaldb.find_cases(WAVEFORM_TRACKS)

print(f"Total cases with required waveform tracks: {len(eligible_ids)}")

# Select only the requested number of cases
selected_ids = eligible_ids[:N_CASES]

print(f"Cases selected for this run: {len(selected_ids)}")
print("Case IDs:", selected_ids)


# 5. Helper function: MAP quality analysis


def calculate_map_quality(map_numeric, interval_seconds=1):
    """
    Evaluate ART_MBP availability without assuming that every
    NaN on a 1-second regular grid represents poor-quality data.

    Returns temporal density and gap statistics for valid MAP
    measurements.
    """

    # Handle empty arrays
    if map_numeric is None or map_numeric.size == 0:
        return {
            "map_valid_count": 0,
            "map_valid_pct": 0.0,
            "map_measurements_per_minute": 0.0,
            "map_median_gap_sec": np.nan,
            "map_p95_gap_sec": np.nan,
            "map_max_gap_sec": np.nan,
            "map_coverage_span_pct": 0.0,
            "map_first_valid_sec": np.nan,
            "map_last_valid_sec": np.nan,
        }

    # Convert to simple 1-D array
    values = map_numeric[:, 0]

    # Identify actual MAP measurements
    valid_mask = ~np.isnan(values)
    valid_indices = np.flatnonzero(valid_mask)

    total_points = len(values)
    valid_count = len(valid_indices)

    # Percentage of the regular 1-second grid containing a value
    valid_pct = (
        valid_count / total_points * 100
        if total_points > 0
        else 0.0
    )

    total_duration_minutes = (
        total_points * interval_seconds / 60
        if total_points > 0
        else 0
    )

    measurements_per_minute = (
        valid_count / total_duration_minutes
        if total_duration_minutes > 0
        else 0.0
    )

    
    # Calculate spacing between valid MAP observations
    

    if valid_count >= 2:

        # Summarize the observed levels and variability for this set of measurements.
        gaps_seconds = (
            np.diff(valid_indices) * interval_seconds
        )

        # Summarize the observed levels and variability for this set of measurements.
        median_gap = float(np.median(gaps_seconds))
        p95_gap = float(np.percentile(gaps_seconds, 95))
        max_gap = float(np.max(gaps_seconds))

        first_valid_sec = float(
            valid_indices[0] * interval_seconds
        )

        last_valid_sec = float(
            valid_indices[-1] * interval_seconds
        )

        # How much of the total case timeline is spanned by MAP data?
        coverage_span_seconds = (
            (valid_indices[-1] - valid_indices[0] + 1)
            * interval_seconds
        )

        total_duration_seconds = (
            total_points * interval_seconds
        )

        coverage_span_pct = (
            coverage_span_seconds
            / total_duration_seconds
            * 100
        )

    elif valid_count == 1:

        median_gap = np.nan
        p95_gap = np.nan
        max_gap = np.nan

        first_valid_sec = float(
            valid_indices[0] * interval_seconds
        )

        last_valid_sec = first_valid_sec

        coverage_span_pct = 0.0

    else:

        median_gap = np.nan
        p95_gap = np.nan
        max_gap = np.nan
        first_valid_sec = np.nan
        last_valid_sec = np.nan
        coverage_span_pct = 0.0

    return {
        "map_valid_count": valid_count,
        "map_valid_pct": valid_pct,
        "map_measurements_per_minute": measurements_per_minute,
        "map_median_gap_sec": median_gap,
        "map_p95_gap_sec": p95_gap,
        "map_max_gap_sec": max_gap,
        "map_coverage_span_pct": coverage_span_pct,
        "map_first_valid_sec": first_valid_sec,
        "map_last_valid_sec": last_valid_sec,
    }


# 6. Helper function: inspect one VitalDB case


def inspect_case(caseid):

    try:

        
        # Load waveforms at 100 Hz
        

        # Load the verified case with separate waveform and numeric-MAP time axes.
        wave = vitaldb.load_case(
            caseid,
            WAVEFORM_TRACKS,
            interval=1 / WAVEFORM_FS
        )

        
        # Load numerical MAP onto a 1-second grid
        

        # Load the verified case with separate waveform and numeric-MAP time axes.
        map_numeric = vitaldb.load_case(
            caseid,
            MAP_TRACK,
            interval=MAP_INTERVAL
        )

        
        # Case duration
        

        duration_seconds = (
            wave.shape[0] / WAVEFORM_FS
        )

        duration_minutes = (
            duration_seconds / 60
        )

        
        # Waveform missingness
        

        missing_art_pct = (
            np.isnan(wave[:, 0]).mean() * 100
        )

        missing_ecg_pct = (
            np.isnan(wave[:, 1]).mean() * 100
        )

        missing_ppg_pct = (
            np.isnan(wave[:, 2]).mean() * 100
        )

        
        # MAP quality
        

        map_quality = calculate_map_quality(
            map_numeric,
            interval_seconds=MAP_INTERVAL
        )

        has_map = (
            map_quality["map_valid_count"] > 0
        )

        
        # Return manifest row
        

        result = {
            "caseid": caseid,

            "duration_seconds": duration_seconds,
            "duration_minutes": duration_minutes,

            "has_art": not np.all(np.isnan(wave[:, 0])),
            "has_ecg": not np.all(np.isnan(wave[:, 1])),
            "has_ppg": not np.all(np.isnan(wave[:, 2])),
            "has_map": has_map,

            "missing_art_pct": missing_art_pct,
            "missing_ecg_pct": missing_ecg_pct,
            "missing_ppg_pct": missing_ppg_pct,

            "processing_status": "checked"
        }

        # Add MAP quality metrics
        result.update(map_quality)

        return result

    except Exception as e:

        return {
            "caseid": caseid,

            "duration_seconds": np.nan,
            "duration_minutes": np.nan,

            "has_art": False,
            "has_ecg": False,
            "has_ppg": False,
            "has_map": False,

            "missing_art_pct": np.nan,
            "missing_ecg_pct": np.nan,
            "missing_ppg_pct": np.nan,

            "map_valid_count": 0,
            "map_valid_pct": 0.0,
            "map_measurements_per_minute": 0.0,
            "map_median_gap_sec": np.nan,
            "map_p95_gap_sec": np.nan,
            "map_max_gap_sec": np.nan,
            "map_coverage_span_pct": 0.0,
            "map_first_valid_sec": np.nan,
            "map_last_valid_sec": np.nan,

            "processing_status": f"error: {e}"
        }


# 7. Process selected cohort


rows = []

for position, caseid in enumerate(selected_ids, start=1):

    print(
        f"Checking case {position}/{len(selected_ids)} "
        f"(caseid={caseid})..."
    )

    result = inspect_case(caseid)

    rows.append(result)

# Convert into DataFrame
manifest = pd.DataFrame(rows)


# 8. Add provisional usability criteria


manifest["usable"] = (

    # Adequate case duration
    (manifest["duration_minutes"] >= MIN_DURATION_MINUTES)

    # Required waveforms actually contain data
    & manifest["has_art"]
    & manifest["has_ecg"]
    & manifest["has_ppg"]

    # Waveform completeness
    & (manifest["missing_art_pct"] < MAX_ART_MISSING_PCT)
    & (manifest["missing_ecg_pct"] < MAX_ECG_MISSING_PCT)
    & (manifest["missing_ppg_pct"] < MAX_PPG_MISSING_PCT)

    # MAP must exist
    & manifest["has_map"]

    # Enough actual MAP observations
    & (manifest["map_valid_count"] >= MIN_MAP_MEASUREMENTS)

    # Reasonable MAP observation density
    & (
        manifest["map_measurements_per_minute"]
        >= MIN_MAP_MEASUREMENTS_PER_MINUTE
    )

    # MAP should span most of the surgical recording
    & (
        manifest["map_coverage_span_pct"]
        >= MIN_MAP_COVERAGE_SPAN_PCT
    )

    # Typical gaps should not be excessively large
    & (
        manifest["map_median_gap_sec"]
        <= MAX_MAP_MEDIAN_GAP_SEC
    )
)

# Placeholder for eventual train / validation / test assignment
manifest["split"] = ""


# 9. Display full manifest


print("\nFULL MANIFEST")
display(manifest)


# 10. Display compact quality summary


summary_columns = [
    "caseid",
    "duration_minutes",

    "missing_art_pct",
    "missing_ecg_pct",
    "missing_ppg_pct",

    "map_valid_count",
    "map_valid_pct",
    "map_measurements_per_minute",

    "map_median_gap_sec",
    "map_p95_gap_sec",
    "map_max_gap_sec",

    "map_coverage_span_pct",

    "usable",
    "processing_status"
]

print("\nQUALITY SUMMARY")
display(manifest[summary_columns])


# 11. Save manifest


csv_path = (
    MANIFEST_DIR /
    f"cohort_manifest_{N_CASES}_cases.csv"
)

# Save the comparison table so the same measurements can be inspected later.
parquet_path = (
    MANIFEST_DIR /
    f"cohort_manifest_{N_CASES}_cases.parquet"
)

# Write the verified values in the pipeline's Parquet table format.
manifest.to_csv(
    csv_path,
    index=False
)

manifest.to_parquet(
    parquet_path,
    index=False
)


# 12. Report results


# Display the recorded values for this step.
print("\n================================================")
print("COHORT CHECK COMPLETE")
# Display the recorded values for this step.
print("================================================")

print(f"Cases requested: {N_CASES}")
# Display the recorded values for this step.
print(f"Cases processed: {len(manifest)}")

# Display the recorded values for this step.
print(
    "Cases successfully checked:",
    (manifest["processing_status"] == "checked").sum()
)

# Display the recorded values for this step.
print(
    "Cases provisionally usable:",
    manifest["usable"].sum()
)

# Display the recorded values for this step.
print(
    "Cases provisionally excluded:",
    (~manifest["usable"]).sum()
)

# Display the recorded values for this step.
print("\nSaved CSV:")
print(csv_path)

print("\nSaved Parquet:")
print(parquet_path)


# 13. Show exclusion reasons


# Display the recorded values for this step.
print("\n================================================")
print("SCREENING DETAILS")
print("================================================")

for _, row in manifest.iterrows():

    # Exclude or flag missing values before using these measurements.
    if row["usable"]:
        # Display the recorded values for this step.
        print(
            f"Case {row['caseid']}: USABLE"
        )

    else:

        reasons = []

        if row["duration_minutes"] < MIN_DURATION_MINUTES:
            reasons.append("recording too short")

        if not row["has_art"]:
            reasons.append("ART unavailable")

        elif row["missing_art_pct"] >= MAX_ART_MISSING_PCT:
            reasons.append("excessive ART missingness")

        if not row["has_ecg"]:
            reasons.append("ECG unavailable")

        elif row["missing_ecg_pct"] >= MAX_ECG_MISSING_PCT:
            reasons.append("excessive ECG missingness")

        if not row["has_ppg"]:
            reasons.append("PPG unavailable")

        elif row["missing_ppg_pct"] >= MAX_PPG_MISSING_PCT:
            reasons.append("excessive PPG missingness")

        if not row["has_map"]:
            reasons.append("MAP unavailable")

        else:

            if row["map_valid_count"] < MIN_MAP_MEASUREMENTS:
                reasons.append("too few MAP measurements")

            if (
                row["map_measurements_per_minute"]
                < MIN_MAP_MEASUREMENTS_PER_MINUTE
            ):
                reasons.append("MAP density too low")

            if (
                row["map_coverage_span_pct"]
                < MIN_MAP_COVERAGE_SPAN_PCT
            ):
                reasons.append("MAP does not span enough of case")

            if (
                pd.notna(row["map_median_gap_sec"])
                and row["map_median_gap_sec"]
                > MAX_MAP_MEDIAN_GAP_SEC
            ):
                reasons.append("MAP sampling gaps too large")

        print(
            f"Case {row['caseid']}: EXCLUDED -> "
            + ", ".join(reasons)
        )

## Assess candidate surgeries

Inspect candidate recordings, apply modality and quality criteria, and save the usable-cohort manifests for waveform ingestion.


In [ ]:
# Load the libraries and project settings needed for this step.
import vitaldb
import pandas as pd
import numpy as np

from pathlib import Path
import tempfile
import shutil
import time



# STAGE D :  FULL ELIGIBLE COHORT
# Automatically resumes from latest saved checkpoint




# 1. PROJECT PATHS


# Create the destination directory before writing any outputs.
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())

# Create the destination directory before writing any outputs.
MANIFEST_DIR = PROJECT / "data" / "manifests"

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LATEST_CHECKPOINT_PATH = (
    MANIFEST_DIR / "stage_d_checkpoint.csv"
)


# Display the recorded values for this step.
print("Project root:")
print(PROJECT)

# Display the recorded values for this step.
print("\nManifest directory:")
print(MANIFEST_DIR)

# Display the recorded values for this step.
print("\nManifest directory exists:")
print(MANIFEST_DIR.exists())

# Display the recorded values for this step.
print("\nExpected checkpoint:")
# Display the recorded values for this step.
print(LATEST_CHECKPOINT_PATH)

print("\nCheckpoint exists:")
print(LATEST_CHECKPOINT_PATH.exists())

print()



# 2. SIGNAL SETTINGS


WAVEFORM_TRACKS = [
    "ART",
    "ECG_II",
    "PLETH"
]

MAP_TRACK = [
    "ART_MBP"
]

WAVEFORM_FS = 100
MAP_INTERVAL = 1



# 3. PROVISIONAL COHORT SELECTION THRESHOLDS
# Same thresholds used during Stage C


MIN_DURATION_MINUTES = 30

MAX_ART_MISSING_PCT = 20
MAX_ECG_MISSING_PCT = 20
MAX_PPG_MISSING_PCT = 20

MIN_MAP_MEASUREMENTS = 300
MIN_MAP_MEASUREMENTS_PER_MINUTE = 10
MIN_MAP_COVERAGE_SPAN_PCT = 80
MAX_MAP_MEDIAN_GAP_SEC = 10



# 4. CHECKPOINT SETTINGS


CHECKPOINT_EVERY = 100



# 5. FIND ALL ELIGIBLE CASES


eligible_ids = vitaldb.find_cases(
    WAVEFORM_TRACKS
)

selected_ids = [
    int(caseid)
    for caseid in eligible_ids
]

TOTAL_CASES = len(selected_ids)


# Display the recorded values for this step.
print("========================================")
print("STAGE D — FULL COHORT")
print("========================================")

# Display the recorded values for this step.
print(
    f"Total eligible cases: {TOTAL_CASES}"
)

print()



# 6. MAP QUALITY HELPER


def calculate_map_quality(
    map_numeric,
    interval_seconds=1
):

    if (
        map_numeric is None
        or map_numeric.size == 0
    ):

        return {

            "map_valid_count": 0,

            "map_valid_pct": 0.0,

            "map_measurements_per_minute": 0.0,

            "map_median_gap_sec": np.nan,

            "map_p95_gap_sec": np.nan,

            "map_max_gap_sec": np.nan,

            "map_coverage_span_pct": 0.0,

            "map_first_valid_sec": np.nan,

            "map_last_valid_sec": np.nan,
        }


    # Calculate the observed values for the following step.
    values = map_numeric[:, 0]

    valid_mask = ~np.isnan(
        values
    )

    valid_indices = np.flatnonzero(
        valid_mask
    )

    total_points = len(
        values
    )

    valid_count = len(
        valid_indices
    )


    
    # Percentage of the 1-second MAP grid containing values
    

    valid_pct = (

        valid_count
        / total_points
        * 100

        if total_points > 0

        else 0.0
    )


    
    # Duration represented by MAP array
    

    total_duration_minutes = (

        total_points
        * interval_seconds
        / 60

        if total_points > 0

        else 0.0
    )


    
    # Actual MAP measurement density
    

    measurements_per_minute = (

        valid_count
        / total_duration_minutes

        if total_duration_minutes > 0

        else 0.0
    )


    
    # MAP timing / coverage statistics
    

    if valid_count >= 2:

        gaps_seconds = (

            np.diff(
                valid_indices
            )

            * interval_seconds
        )


        # Summarize the observed levels and variability for this set of measurements.
        median_gap = float(
            np.median(
                gaps_seconds
            )
        )


        # Summarize the observed levels and variability for this set of measurements.
        p95_gap = float(
            np.percentile(
                gaps_seconds,
                95
            )
        )


        max_gap = float(
            np.max(
                gaps_seconds
            )
        )


        first_valid_sec = float(

            valid_indices[0]
            * interval_seconds
        )


        last_valid_sec = float(

            valid_indices[-1]
            * interval_seconds
        )


        coverage_span_seconds = (

            (
                valid_indices[-1]
                - valid_indices[0]
                + 1
            )

            * interval_seconds
        )


        total_duration_seconds = (

            total_points
            * interval_seconds
        )


        coverage_span_pct = (

            coverage_span_seconds
            / total_duration_seconds
            * 100
        )


    elif valid_count == 1:

        median_gap = np.nan

        p95_gap = np.nan

        max_gap = np.nan


        first_valid_sec = float(

            valid_indices[0]
            * interval_seconds
        )


        last_valid_sec = (
            first_valid_sec
        )


        coverage_span_pct = 0.0


    else:

        median_gap = np.nan

        p95_gap = np.nan

        max_gap = np.nan

        first_valid_sec = np.nan

        last_valid_sec = np.nan

        coverage_span_pct = 0.0


    return {

        "map_valid_count":
            valid_count,

        "map_valid_pct":
            valid_pct,

        "map_measurements_per_minute":
            measurements_per_minute,

        "map_median_gap_sec":
            median_gap,

        "map_p95_gap_sec":
            p95_gap,

        "map_max_gap_sec":
            max_gap,

        "map_coverage_span_pct":
            coverage_span_pct,

        "map_first_valid_sec":
            first_valid_sec,

        "map_last_valid_sec":
            last_valid_sec,
    }



# 7. INSPECT ONE CASE


def inspect_case(caseid):

    try:

        
        # Load ART, ECG and PPG at 100 Hz
        

        # Load the verified case with separate waveform and numeric-MAP time axes.
        wave = vitaldb.load_case(

            caseid,

            WAVEFORM_TRACKS,

            interval=1 / WAVEFORM_FS
        )


        
        # Load numeric MAP
        

        # Load the verified case with separate waveform and numeric-MAP time axes.
        map_numeric = vitaldb.load_case(

            caseid,

            MAP_TRACK,

            interval=MAP_INTERVAL
        )


        
        # Duration
        

        duration_seconds = (

            wave.shape[0]
            / WAVEFORM_FS
        )


        duration_minutes = (

            duration_seconds
            / 60
        )


        
        # Waveform missingness
        

        missing_art_pct = (

            np.isnan(
                wave[:, 0]
            ).mean()

            * 100
        )


        missing_ecg_pct = (

            np.isnan(
                wave[:, 1]
            ).mean()

            * 100
        )


        missing_ppg_pct = (

            np.isnan(
                wave[:, 2]
            ).mean()

            * 100
        )


        
        # MAP quality
        

        map_quality = (
            calculate_map_quality(

                map_numeric,

                interval_seconds=MAP_INTERVAL
            )
        )


        
        # Build result
        

        result = {

            "caseid":
                int(caseid),

            "duration_seconds":
                duration_seconds,

            "duration_minutes":
                duration_minutes,


            "has_art":

                not np.all(
                    np.isnan(
                        wave[:, 0]
                    )
                ),


            "has_ecg":

                not np.all(
                    np.isnan(
                        wave[:, 1]
                    )
                ),


            "has_ppg":

                not np.all(
                    np.isnan(
                        wave[:, 2]
                    )
                ),


            "has_map":

                (
                    map_quality[
                        "map_valid_count"
                    ]
                    > 0
                ),


            "missing_art_pct":
                missing_art_pct,

            "missing_ecg_pct":
                missing_ecg_pct,

            "missing_ppg_pct":
                missing_ppg_pct,


            "processing_status":
                "checked"
        }


        result.update(
            map_quality
        )


        return result


    except Exception as e:

        
        # Do not terminate Stage D because one case failed.
        # Record the error instead.
        

        return {

            "caseid":
                int(caseid),

            "duration_seconds":
                np.nan,

            "duration_minutes":
                np.nan,

            "has_art":
                False,

            "has_ecg":
                False,

            "has_ppg":
                False,

            "has_map":
                False,

            "missing_art_pct":
                np.nan,

            "missing_ecg_pct":
                np.nan,

            "missing_ppg_pct":
                np.nan,

            "map_valid_count":
                0,

            "map_valid_pct":
                0.0,

            "map_measurements_per_minute":
                0.0,

            "map_median_gap_sec":
                np.nan,

            "map_p95_gap_sec":
                np.nan,

            "map_max_gap_sec":
                np.nan,

            "map_coverage_span_pct":
                0.0,

            "map_first_valid_sec":
                np.nan,

            "map_last_valid_sec":
                np.nan,

            "processing_status":
                f"error: {e}"
        }



# 8. FIND LATEST CHECKPOINT

# Looks for:

# stage_d_checkpoint.csv
# stage_d_checkpoint_1900.csv
# stage_d_checkpoint_2000.csv
# ...

# If more than one exists, the checkpoint containing the most
# processed cases is selected.


def find_latest_checkpoint():

    candidates = list(

        MANIFEST_DIR.glob(
            "stage_d_checkpoint*.csv"
        )
    )


    if not candidates:

        return None, None


    print(
        f"Found {len(candidates)} "
        f"checkpoint file(s)."
    )


    valid_checkpoints = []


    # Process the selected records in order and retain their individual results.
    for path in candidates:

        try:

            # Read the saved table and retain its named columns.
            df = pd.read_csv(
                path
            )


            
            # Must contain case IDs
            

            if "caseid" not in df.columns:

                print(
                    f"Ignoring {path.name}: "
                    f"no caseid column."
                )

                continue


            
            # Must contain at least one row
            

            if len(df) == 0:

                print(
                    f"Ignoring {path.name}: "
                    f"empty file."
                )

                continue


            
            # Convert case IDs to integers
            

            df["caseid"] = (

                df["caseid"]
                .astype(int)
            )


            
            # Reject duplicate rows inside a checkpoint
            

            if df["caseid"].duplicated().any():

                print(
                    f"Ignoring {path.name}: "
                    f"duplicate case IDs."
                )

                continue


            
            # Cannot contain more rows than complete cohort
            

            if len(df) > TOTAL_CASES:

                print(
                    f"Ignoring {path.name}: "
                    f"contains more rows "
                    f"than the full cohort."
                )

                continue


            valid_checkpoints.append(

                (
                    len(df),

                    path.stat().st_mtime,

                    path,

                    df
                )
            )


            # Display the recorded values for this step.
            print(
                f"Valid checkpoint: "
                f"{path.name} "
                f"({len(df)} cases)"
            )


        except Exception as e:

            # Display the recorded values for this step.
            print(
                f"Could not read "
                f"{path.name}: {e}"
            )


    if not valid_checkpoints:

        return None, None


    
    # Largest checkpoint wins.
    
    # If two contain the same number of cases,
    # newest modification time wins.
    

    valid_checkpoints.sort(

        key=lambda item: (

            item[0],

            item[1]
        ),

        reverse=True
    )


    _, _, best_path, best_df = (

        valid_checkpoints[0]
    )


    return (
        best_path,
        best_df
    )



# 9. SAFE CHECKPOINT WRITER

# Saves locally first and then copies into Google Drive.


def save_checkpoint(rows):

    # Collect the per-record values into a table for comparison.
    case_count = len(
        rows
    )


    checkpoint_df = pd.DataFrame(
        rows
    )


    
    # Numbered checkpoint
    

    numbered_path = (

        MANIFEST_DIR

        / f"stage_d_checkpoint_{case_count}.csv"
    )


    
    # Temporary local file
    

    temp_dir = Path(
        tempfile.gettempdir()
    )


    temp_path = (

        temp_dir

        / f"stage_d_checkpoint_{case_count}_temp.csv"
    )


    # Save the comparison table so the same measurements can be inspected later.
    checkpoint_df.to_csv(

        temp_path,

        index=False
    )


    
    # Copy temporary file to Google Drive
    
    # Retry up to 3 times if Drive is temporarily busy.
    

    save_success = False


    # Accumulate the calculation over the required samples or iterations.
    for attempt in range(
        1,
        4
    ):

        try:

            shutil.copyfile(

                temp_path,

                numbered_path
            )


            save_success = True

            break


        except OSError as e:

            # Display the recorded values for this step.
            print()

            print(
                f"Checkpoint save attempt "
                f"{attempt}/3 failed."
            )

            # Display the recorded values for this step.
            print(
                e
            )


            if attempt < 3:

                # Display the recorded values for this step.
                print(
                    "Waiting 3 seconds "
                    "before retrying..."
                )

                time.sleep(
                    3
                )


    
    # Do not continue if numbered checkpoint cannot be saved.
    

    if not save_success:

        raise OSError(

            "Could not save checkpoint to "
            "Google Drive. "

            f"Temporary checkpoint remains here: "
            f"{temp_path}"
        )


    
    # Update generic latest checkpoint
    
    # This deliberately overwrites:
    
    # stage_d_checkpoint.csv
    
    # The numbered checkpoint remains as backup.
    

    try:

        shutil.copyfile(

            numbered_path,

            LATEST_CHECKPOINT_PATH
        )


    except OSError as e:

        # Display the recorded values for this step.
        print()

        print(
            "WARNING:"
        )

        print(
            "The numbered checkpoint was saved, "
            "but stage_d_checkpoint.csv could "
            "not be updated."
        )

        # Display the recorded values for this step.
        print(
            e
        )


    
    # Remove local temporary file
    

    try:

        # Clean up the working files owned by this operation.
        temp_path.unlink(
            missing_ok=True
        )

    except Exception:

        pass


    # Display the recorded values for this step.
    print()

    print(
        f"Checkpoint saved after "
        f"{case_count} cases"
    )

    # Display the recorded values for this step.
    print(
        f" -> {numbered_path.name}"
    )

    print()



# 10. LOAD LATEST CHECKPOINT


checkpoint_path, checkpoint_df = (
    find_latest_checkpoint()
)


if checkpoint_df is None:

    # Display the recorded values for this step.
    print()

    print(
        "No checkpoint found."
    )

    print(
        "Starting Stage D from case 1."
    )

    print()


    rows = []

    processed_ids = set()


else:

    
    # Restore previously processed rows
    

    rows = checkpoint_df.to_dict(
        "records"
    )


    processed_ids = set(

        checkpoint_df[
            "caseid"
        ]

        .astype(int)

        .tolist()
    )


    # Display the recorded values for this step.
    print()

    print(
        "========================================"
    )

    print(
        "CHECKPOINT LOADED"
    )

    # Display the recorded values for this step.
    print(
        "========================================"
    )


    print(
        "Checkpoint file:"
    )

    # Display the recorded values for this step.
    print(
        checkpoint_path
    )


    print()


    # Display the recorded values for this step.
    print(
        f"Cases safely loaded: "
        f"{len(rows)}"
    )


    print()



# 11. DETERMINE REMAINING CASES

# Every case ID already present in the checkpoint is skipped.

# Anything processed after the last successful checkpoint but
# lost during the crash is recomputed.


remaining_ids = [

    caseid

    for caseid in selected_ids

    if caseid not in processed_ids
]


# Display the recorded values for this step.
print(
    f"Total eligible cases: "
    f"{TOTAL_CASES}"
)

# Display the recorded values for this step.
print(
    f"Already processed: "
    f"{len(processed_ids)}"
)

# Display the recorded values for this step.
print(
    f"Remaining to process: "
    f"{len(remaining_ids)}"
)

print()


if remaining_ids:

    # Display the recorded values for this step.
    print(
        f"Next readout:"
    )

    print(

        f"Checking "
        f"{len(rows) + 1}/"
        f"{TOTAL_CASES} "
        f"— case {remaining_ids[0]}"
    )

    print()



# 12. RUN / RESUME FULL COHORT


for caseid in remaining_ids:

    case_number = (

        len(rows)
        + 1
    )


    # Display the recorded values for this step.
    print(

        f"Checking "
        f"{case_number}/"
        f"{TOTAL_CASES} "
        f"— case {caseid}"
    )


    result = inspect_case(
        caseid
    )


    rows.append(
        result
    )


    
    # Save every 100 TOTAL cases.
    
    # Example if 1800 loaded:
    
    # 1900 -> save
    # 2000 -> save
    # 2100 -> save
    # ...
    

    if (

        len(rows)
        % CHECKPOINT_EVERY

        == 0
    ):

        save_checkpoint(
            rows
        )



# 13. SAVE FINAL CHECKPOINT

# Example:

# 3458 is not divisible by 100, so we still save 3458.


if len(rows) > 0:

    if (

        len(rows)
        % CHECKPOINT_EVERY

        != 0
    ):

        save_checkpoint(
            rows
        )



# 14. BUILD FINAL MANIFEST


# Collect the per-record values into a table for comparison.
manifest = pd.DataFrame(
    rows
)



# 15. FINAL DATA INTEGRITY CHECKS


duplicate_count = (

    manifest[
        "caseid"
    ]

    .duplicated()

    .sum()
)


if duplicate_count > 0:

    raise RuntimeError(

        f"Final manifest contains "
        f"{duplicate_count} duplicate case IDs."
    )


if len(manifest) != TOTAL_CASES:

    # Display the recorded values for this step.
    print()

    print(
        "WARNING:"
    )

    print(

        f"Expected {TOTAL_CASES} cases, "
        f"but the final manifest contains "
        f"{len(manifest)} cases."
    )

    print()



# 16. APPLY PROVISIONAL USABILITY RULES


manifest["usable"] = (

    (
        manifest[
            "duration_minutes"
        ]

        >= MIN_DURATION_MINUTES
    )


    & manifest[
        "has_art"
    ]


    & manifest[
        "has_ecg"
    ]


    & manifest[
        "has_ppg"
    ]


    & (
        manifest[
            "missing_art_pct"
        ]

        < MAX_ART_MISSING_PCT
    )


    & (
        manifest[
            "missing_ecg_pct"
        ]

        < MAX_ECG_MISSING_PCT
    )


    & (
        manifest[
            "missing_ppg_pct"
        ]

        < MAX_PPG_MISSING_PCT
    )


    & manifest[
        "has_map"
    ]


    & (
        manifest[
            "map_valid_count"
        ]

        >= MIN_MAP_MEASUREMENTS
    )


    & (
        manifest[
            "map_measurements_per_minute"
        ]

        >= MIN_MAP_MEASUREMENTS_PER_MINUTE
    )


    & (
        manifest[
            "map_coverage_span_pct"
        ]

        >= MIN_MAP_COVERAGE_SPAN_PCT
    )


    & (
        manifest[
            "map_median_gap_sec"
        ]

        <= MAX_MAP_MEDIAN_GAP_SEC
    )
)



# 17. PLACEHOLDER FOR FUTURE TRAIN / VALIDATION / TEST SPLIT


manifest["split"] = ""



# 18. ORDER FINAL MANIFEST ACCORDING TO CURRENT COHORT ORDER


case_order = {

    caseid: position

    for position, caseid

    in enumerate(
        selected_ids
    )
}


manifest["_case_order"] = (

    manifest[
        "caseid"
    ]

    .map(
        case_order
    )
)


# Order the measurements before matching intervals or comparing rows.
manifest = (

    manifest

    .sort_values(
        "_case_order"
    )

    .drop(
        columns=[
            "_case_order"
        ]
    )

    .reset_index(
        drop=True
    )
)



# 19. SAVE FINAL FULL-COHORT MANIFEST


csv_path = (

    MANIFEST_DIR

    / "stage_d_full_cohort.csv"
)


parquet_path = (

    MANIFEST_DIR

    / "stage_d_full_cohort.parquet"
)


# Save the comparison table so the same measurements can be inspected later.
manifest.to_csv(

    csv_path,

    index=False
)


# Write the verified values in the pipeline's Parquet table format.
manifest.to_parquet(

    parquet_path,

    index=False
)



# 20. SAVE USABLE COHORT SEPARATELY


usable_manifest = (

    manifest[

        manifest[
            "usable"
        ]
    ]

    .copy()
)


usable_csv_path = (

    MANIFEST_DIR

    / "usable_cohort.csv"
)


usable_parquet_path = (

    MANIFEST_DIR

    / "usable_cohort.parquet"
)


# Save the comparison table so the same measurements can be inspected later.
usable_manifest.to_csv(

    usable_csv_path,

    index=False
)


# Write the verified values in the pipeline's Parquet table format.
usable_manifest.to_parquet(

    usable_parquet_path,

    index=False
)



# 21. FINAL SUMMARY


# Display the recorded values for this step.
print()

# Display the recorded values for this step.
print(
    "========================================"
)

# Display the recorded values for this step.
print(
    "STAGE D COMPLETE"
)

# Display the recorded values for this step.
print(
    "========================================"
)


print(
    "Total track-eligible cases:",
    len(manifest)
)


# Display the recorded values for this step.
print(
    "Successfully checked:",

    (
        manifest[
            "processing_status"
        ]

        == "checked"
    ).sum()
)


# Display the recorded values for this step.
print(
    "Cases with processing errors:",

    (
        manifest[
            "processing_status"
        ]

        != "checked"
    ).sum()
)


# Display the recorded values for this step.
print(
    "Provisionally usable:",

    int(

        manifest[
            "usable"
        ].sum()
    )
)


# Display the recorded values for this step.
print(
    "Provisionally excluded:",

    int(

        (
            ~manifest[
                "usable"
            ]
        ).sum()
    )
)


# Display the recorded values for this step.
print()

# Display the recorded values for this step.
print(
    "Final full-cohort CSV:"
)

# Display the recorded values for this step.
print(
    csv_path
)


# Display the recorded values for this step.
print()

# Display the recorded values for this step.
print(
    "Final full-cohort Parquet:"
)

# Display the recorded values for this step.
print(
    parquet_path
)


# Display the recorded values for this step.
print()

# Display the recorded values for this step.
print(
    "Usable-cohort CSV:"
)

# Display the recorded values for this step.
print(
    usable_csv_path
)


# Display the recorded values for this step.
print()

# Display the recorded values for this step.
print(
    "Usable-cohort Parquet:"
)

print(
    usable_parquet_path
)